# Week 1 Day 3 — Autograd: requires_grad, backward, gradients
**Jul 1, 2026**

Autograd is PyTorch's automatic differentiation engine. Understanding it deeply means you'll never be confused by training loop bugs again.

## Part 1: `requires_grad` basics

In [1]:
import torch

# By default, tensors do NOT track gradients
x = torch.tensor([2.0, 3.0])
print('requires_grad default:', x.requires_grad)   # False

# Turn on gradient tracking
x = torch.tensor([2.0, 3.0], requires_grad=True)
print('requires_grad on:', x.requires_grad)        # True

requires_grad default: False
requires_grad on: True


In [2]:
# Tensors created from operations on tracked tensors also track
y = x * 2
print('y.requires_grad:', y.requires_grad)   # True
print('y.grad_fn:', y.grad_fn)               # MulBackward0 — PyTorch remembers the op

# Leaf vs non-leaf
print('x is leaf:', x.is_leaf)   # True  — you created it
print('y is leaf:', y.is_leaf)   # False — result of an op

y.requires_grad: True
y.grad_fn: <MulBackward0 object at 0x10570f1c0>
x is leaf: True
y is leaf: False


## Part 2: Computational graph + `backward()`

In [3]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

y = x * 2           # [2, 4, 6]
z = (y ** 2).sum()  # scalar: 4 + 16 + 36 = 56
print('z:', z.item())

# backward() computes dz/dx for all leaf tensors with requires_grad=True
z.backward()

# dz/dx = d/dx sum((2x)^2) = d/dx sum(4x^2) = 8x
print('x.grad:', x.grad)   # tensor([ 8., 16., 24.])

z: 56.0
x.grad: tensor([ 8., 16., 24.])


**Key rule:** `backward()` only works on scalars. This is why loss functions always return a scalar.

## Part 3: Gradient accumulation + `zero_grad()`

In [4]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# First pass
loss1 = (x ** 2).sum()
loss1.backward()
print('After 1st backward:', x.grad)           # [2, 4, 6]

# Second pass WITHOUT zeroing — gradients ACCUMULATE
loss2 = (x ** 2).sum()
loss2.backward()
print('After 2nd backward (no zero):', x.grad) # [4, 8, 12] — doubled!

# Zero before next pass
x.grad.zero_()
loss3 = (x ** 2).sum()
loss3.backward()
print('After zeroing + 3rd backward:', x.grad) # [2, 4, 6] — correct

After 1st backward: tensor([2., 4., 6.])
After 2nd backward (no zero): tensor([ 4.,  8., 12.])
After zeroing + 3rd backward: tensor([2., 4., 6.])


This is why `optimizer.zero_grad()` must be called before every backward pass in your training loop.

## Part 4: `torch.no_grad()` — inference mode

In [5]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# During inference: no graph needed — saves memory and compute
with torch.no_grad():
    y = x * 2
    print('inside no_grad, y.requires_grad:', y.requires_grad)   # False

# detach() — pull a tensor out of the graph entirely
z = x * 3
z_detached = z.detach()
print('z_detached.requires_grad:', z_detached.requires_grad)     # False
# use detach() for: logging losses, computing metrics, converting to numpy

inside no_grad, y.requires_grad: False
z_detached.requires_grad: False


**Rule:** `model.eval()` + `torch.no_grad()` always go together in a validation loop.

## Part 5: Real use case — manual gradient descent

Fit `y = 3x + 2` from noisy data using raw gradient updates (no optimizer yet).

In [10]:
torch.manual_seed(42)

# True parameters
true_w, true_b = 3.0, 2.0

# Generate data
X = torch.randn(50)
y = true_w * X + true_b + 0.1 * torch.randn(50)

# Learnable parameters
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

lr = 0.05
n_epochs = 50

for epoch in range(n_epochs):
    y_pred = w * X + b
    loss = ((y_pred - y) ** 2).mean()

    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    w.grad.zero_()
    b.grad.zero_()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:2d} | loss: {loss.item():.4f} | w: {w.item():.3f} | b: {b.item():.3f}')

print(f'\nTrue:    w={true_w}, b={true_b}')
print(f'Learned: w={w.item():.3f}, b={b.item():.3f}')

Epoch 10 | loss: 1.6778 | w: 2.117 | b: 1.417
Epoch 20 | loss: 0.1539 | w: 2.740 | b: 1.838
Epoch 30 | loss: 0.0211 | w: 2.923 | b: 1.964
Epoch 40 | loss: 0.0096 | w: 2.977 | b: 2.001
Epoch 50 | loss: 0.0085 | w: 2.993 | b: 2.013

True:    w=3.0, b=2.0
Learned: w=2.993, b=2.013


Tomorrow (Day 4) you'll replace `w -= lr * w.grad` with `torch.optim.SGD` — same thing, cleaner API.

## Part 6: Vector-valued `backward()`

In [7]:
# If output is not scalar, pass a gradient vector
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2      # [1, 4, 9] — NOT a scalar

v = torch.ones(3)
y.backward(v)
print('vector backward grad:', x.grad)   # [2, 4, 6]
# equivalent to d(sum(y))/dx — same as if you did y.sum().backward()

vector backward grad: tensor([2., 4., 6.])


## Summary

| Concept | What it does |
|---|---|
| `requires_grad=True` | tells PyTorch to track this tensor |
| `.backward()` | computes all gradients from a scalar output |
| `.grad` | stores dL/d(tensor) after backward |
| `zero_grad()` | MUST call before each backward — grads accumulate |
| `torch.no_grad()` | use during validation/inference — no graph built |
| `.detach()` | pulls tensor out of graph for logging/numpy |

## Try yourself

1. Change Part 2 so `z = (y ** 3).sum()` — compute the gradient by hand first, then verify
2. In Part 5, remove `w.grad.zero_()` and observe what happens to the learned weights
3. In Part 4, try calling `.numpy()` on a tensor that has `requires_grad=True` — what error do you get? Fix it with `.detach()`